In [1]:
import pandas as pd
import numpy as np

In [2]:
def select_best_ensemble_k(model_files, k=3, keep_top_n=7):
    """
    model_files: 字典 {模型名稱: 檔案路徑}
    k: 最終要選出來投票的模型數量
    keep_top_n: 初始篩選時要保留的前幾個模型 (依正確率排序)
    """
    
    results_wrong_vec = {} # 儲存每個模型是否答錯的向量 (1為錯, 0為對)
    accuracies = {}
    
    # 1. 讀取與計算基本數據
    for name, path in model_files.items():
        df = pd.read_excel(path)
        # 確保資料格式一致，避免 string vs int 的比對錯誤
        actual = df['Actual'].astype(str).str.strip()
        predicted = df['Predicted'].fillna(0).astype(int).astype(str).str.strip()
        
        # 計算正確率
        is_correct = (actual == predicted)
        accuracies[name] = is_correct.mean()
        
        # 記錄答錯位置 (True=1, False=0)
        results_wrong_vec[name] = (~is_correct).astype(int).values
    
    # 2. 依照正確率排序並篩選
    # 這裡我們用參數 keep_top_n 來決定保留多少模型
    sorted_by_acc = sorted(accuracies.items(), key=lambda x: x[1], reverse=True)
    top_models = [m[0] for m in sorted_by_acc[:keep_top_n]]
    
    print(f"--- 模型篩選結果 (保留前 {keep_top_n} 名) ---")
    for name in top_models:
        print(f"{name}: {accuracies[name]:.2%}")

    # 3. 計算雙重錯誤率矩陣
    num_questions = len(next(iter(results_wrong_vec.values())))
    de_matrix = pd.DataFrame(index=top_models, columns=top_models)

    for m1 in top_models:
        for m2 in top_models:
            if m1 == m2:
                de_matrix.loc[m1, m2] = np.nan
            else:
                # 兩者同時答錯 (向量相乘或用 &)
                both_wrong = np.sum((results_wrong_vec[m1] == 1) & (results_wrong_vec[m2] == 1))
                de_matrix.loc[m1, m2] = both_wrong / num_questions

    # 儲存矩陣
    de_matrix.to_csv("double_error_matrix.csv")

    # 4. 貪婪演算法選出 K 個模型 (加入偵錯輸出)
    selected = [top_models[0]] 
    remaining = [m for m in top_models if m not in selected]

    print(f"\n--- 迭代選擇過程 ---")
    print(f"Round 1: 初始基準模型 (正確率最高) = {selected[0]}")

    while len(selected) < k and remaining:
        scores = {}
        print(f"\nRound {len(selected) + 1} 評分中...")
        for candidate in remaining:
            # 詳列計算過程
            de_list = [de_matrix.loc[candidate, s] for s in selected]
            avg_de = np.mean(de_list)
            scores[candidate] = avg_de
            print(f"  > 候選 {candidate}: 與 {selected} 的錯誤率分別為 {de_list}, 平均 = {avg_de:.6f}")
        
        best_candidate = min(scores, key=scores.get)
        print(f"決策：選擇平均錯誤率最低的 [{best_candidate}]")
        
        selected.append(best_candidate)
        remaining.remove(best_candidate)
    return selected, de_matrix

In [3]:
# 這裡放入你的模型名稱與對應檔案
MODELS = ["gemma3_12b", "qwen3_8b", "deepseek-r1_8b", "llama3.1_8b", "mistral_7b", "huihui_ai_internlm3-abliterated", 
          "ingu627_phi-4-mini-instruct-q4", "glm4_9b", "myaniu_minicpm4-8b_Q4_K_M"]
# 建立範例檔案路徑 (請確保這些 xlsx 檔案存在)
MODEL_FILE_MAP = {m: f"result_{m}.xlsx" for m in MODELS}

# 執行選擇邏輯
final_models, full_matrix = select_best_ensemble_k(MODEL_FILE_MAP, k=3, keep_top_n=6)
print(f"\n最終建議投票的組合: {final_models}")

--- 模型篩選結果 (保留前 6 名) ---
huihui_ai_internlm3-abliterated: 94.11%
glm4_9b: 93.46%
qwen3_8b: 93.36%
gemma3_12b: 92.86%
myaniu_minicpm4-8b_Q4_K_M: 91.31%
deepseek-r1_8b: 90.41%

--- 迭代選擇過程 ---
Round 1: 初始基準模型 (正確率最高) = huihui_ai_internlm3-abliterated

Round 2 評分中...
  > 候選 glm4_9b: 與 ['huihui_ai_internlm3-abliterated'] 的錯誤率分別為 [np.float64(0.03396603396603397)], 平均 = 0.033966
  > 候選 qwen3_8b: 與 ['huihui_ai_internlm3-abliterated'] 的錯誤率分別為 [np.float64(0.03296703296703297)], 平均 = 0.032967
  > 候選 gemma3_12b: 與 ['huihui_ai_internlm3-abliterated'] 的錯誤率分別為 [np.float64(0.036463536463536464)], 平均 = 0.036464
  > 候選 myaniu_minicpm4-8b_Q4_K_M: 與 ['huihui_ai_internlm3-abliterated'] 的錯誤率分別為 [np.float64(0.03596403596403597)], 平均 = 0.035964
  > 候選 deepseek-r1_8b: 與 ['huihui_ai_internlm3-abliterated'] 的錯誤率分別為 [np.float64(0.03796203796203796)], 平均 = 0.037962
決策：選擇平均錯誤率最低的 [qwen3_8b]

Round 3 評分中...
  > 候選 glm4_9b: 與 ['huihui_ai_internlm3-abliterated', 'qwen3_8b'] 的錯誤率分別為 [np.float64(0.03396603396603397), np

In [4]:
full_matrix

,huihui_ai_internlm3-abliterated,glm4_9b,qwen3_8b,gemma3_12b,myaniu_minicpm4-8b_Q4_K_M,deepseek-r1_8b
huihui_ai_internlm3-abliterated,NaN,0.033966,0.032967,0.036464,0.035964,0.037962
glm4_9b,0.033966,NaN,0.037962,0.03996,0.040959,0.040959
qwen3_8b,0.032967,0.037962,NaN,0.038961,0.04046,0.053447
gemma3_12b,0.036464,0.03996,0.038961,NaN,0.03996,0.040959
myaniu_minicpm4-8b_Q4_K_M,0.035964,0.040959,0.04046,0.03996,NaN,0.047453
deepseek-r1_8b,0.037962,0.040959,0.053447,0.040959,0.047453,NaN
